<a href="https://colab.research.google.com/github/Iberasa3/BlackAndOchre/blob/main/Avispas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vespa velutina macro analysis notebook:

## This section is basic dataframe processing and analysis


In [ ]:
import pandas as pd

# The '\t' separator tells pandas to use the tab key as a delimiter
df = pd.read_csv('avispas_avistamientos.csv', sep='\t')
print(f"Number of columns: {df.shape[1]}")
print(f"Column names: {df.columns.tolist()[:5]}...")

In [ ]:
df['infraspecificEpithet'].unique() # Just cheking how many subespecies there are

In [ ]:
print(df['infraspecificEpithet'].value_counts(dropna=False)) #Checking instances of subespecies

In [ ]:
europe = ['ES', 'FR', 'PT', 'BE', 'IT', 'DE', 'CH', 'LU', 'GB', 'IE', 'NL']
# The analysis is focused on Europe, so we are keeping only European countries.
# Most of the records are European anyway.
df = df[df['countryCode'].isin(europe)]

print(f"Number of records changed from {len(df)} to {len(df)}.")

In [ ]:
# Counting sightings per country and sorting from highest to lowest
country_counts = df['countryCode'].value_counts()

print("Sightings by Country in Europe")
print(country_counts)

In [ ]:
# Cleaning coordinates: removing zero values and nulls
df = df[(df['decimalLatitude'] != 0.0) & (df['decimalLongitude'] != 0.0)]
df = df.dropna(subset=['decimalLatitude', 'decimalLongitude'])

print(f"Original records: {len(df)}")
print(f"Records after removing nulls and null-islands: {len(df)}")

In [ ]:
# Removing instances where uncertainty exceeds 1km.
max_threshold = 1000
df = df[(df['coordinateUncertaintyInMeters'] <= max_threshold) & (df['year'] >= 2005)]

In [ ]:
df = df[df['basisOfRecord'] != 'PRESERVED_SPECIMEN']
print(df['basisOfRecord'].unique())

## Spatial Thinning and Data Independence

We are implementing **spatial thinning** to address spatial autocorrelation. Since we are using a **Random Forest** model, we must satisfy the assumption of independence between observations. Clustered points often reflect sightings from the same colony rather than independent distribution events.

Given that *Vespa velutina* typically forages within a **1 to 1.5 km radius** of its nest, sightings within this range are likely linked to the same location. By applying spatial thinning at this scale, we ensure that:
1. We avoid **oversampling** in highly monitored areas.
2. We treat the project as a **distribution model** (presence/absence) rather than an abundance model.
3. The spatial resolution remains consistent with the species' biological range.



In [ ]:
# Checking for and removing duplicate records based on gbifID
num_duplicates = df.duplicated(subset='gbifID').sum()
print(f"Records with the same gbifID: {num_duplicates}")

# Keeping only the first occurrence of each unique ID
df.drop_duplicates(subset='gbifID', inplace=True)

In [ ]:
# 0.015 degrees of latitude is approximately 1.1 km.
# Each grid cell will be roughly 1.1 km x 1.1 km,
# aligning with the standard 1km resolution for bioclimatic models.
res = 0.015

# Divide by resolution, round to the nearest integer, and multiply back.
df['lat_grid'] = (df['decimalLatitude'] / res).round() * res
df['lon_grid'] = (df['decimalLongitude'] / res).round() * res

print(f"Original record count: {len(df)}")

# Dropping spatial duplicates within the same grid cell
df = df.drop_duplicates(subset=['lat_grid', 'lon_grid'], keep='first').copy()

print(f"Dataset after thinning (1km): {len(df)} records")

df[['decimalLatitude', 'lat_grid', 'decimalLongitude', 'lon_grid']].head()

We have reduced the dataset by nearly 90% by removing extremely close sightings. We are now left with 29,300 records to train our model.

In [ ]:
# Counting how many records exist per country and sorting from highest to lowest.
# These represent the specific grid cells where sightings have occurred.
country_counts = df['countryCode'].value_counts()
total_count = df

print("--- Sightings by Country in Europe ---")
print(country_counts)

## Getting presences and their geographical data


This section handles the integration with Google Earth Engine (GEE) by authenticating and initializing the API. The local dataset is streamlined to include only essential spatial variables (latitude, longitude, and year) and is then converted into an Earth Engine FeatureCollection. This transformation shifts the data from a local Python environment to the cloud, enabling a interactive mapping of Vespa velutina occurrences across Europe

In [ ]:
import sys
import geemap
import ee  # Import the Earth Engine library

# Install geojson library if not already installed
!pip install geojson

# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='vespa-489513')

# Filtering only the required columns for the spatial model
required_columns = ['decimalLatitude', 'decimalLongitude', 'year']
df = df[required_columns].copy()

# Converting the pandas DataFrame to an Earth Engine FeatureCollection
# These will be our spatial points for all subsequent analysis
geospatial_points = geemap.pandas_to_ee(
    df,
    latitude='decimalLatitude',
    longitude='decimalLongitude'
)

"""
UNCOMMENT THIS IF YOU WANT TO SEE THE MAP OF PRESENCES


# Visualizing the points on an interactive map
Map = geemap.Map()
Map.addLayer(geospatial_points, {'color': 'red'}, 'Vespa velutina (Optimized)')
Map.centerObject(geospatial_points, 6)
Map
"""

--------------------------------------------

Given that the existing literature provides clear indicators of the environmental requirements for *Vespa velutina*, we will bypass **Principal Component Analysis (PCA)** in favor of a more interpretable approach. Instead, we will perform a **Multicollinearity Analysis** using the **Variance Inflation Factor (VIF)** to ensure our predictors are independent.
The candidate variables for the model include:

* **Bioclimatic (WorldClim):** Mean annual temperature (**bio01**), Max temperature of the warmest month (**bio05**), Min temperature of the coldest month (**bio06**), and Annual temperature range (**bio07**).
* **Precipitation (WorldClim):** Annual precipitation (**bio12**), Precipitation of the wettest month (**bio13**), Precipitation of the driest month (**bio14**), and Precipitation seasonality (**bio15**).
* **Topography (NASA SRTM):** Elevation and Slope.
* **Land Cover & Infrastructure:** ESA WorldCover (land type) and proximity to roads or urban areas (OpenStreetMap/GEE).

In [ ]:
# PREDICTOR CONFIGURATION

# Define the Area of Interest (AOI) with a 50km buffer
aoi = geospatial_points.geometry().bounds().buffer(50000) #This will be rewritten in the future

# Bioclimatic variables (WorldClim)
bioclim = ee.Image('WORLDCLIM/V1/BIO').clip(aoi)

# Topography variables (SRTM)
srtm = ee.Image('USGS/SRTMGL1_003').clip(aoi)
elevation = srtm.select('elevation')
slope = ee.Terrain.slope(srtm).rename('slope')

# Land cover (ESA WorldCover)
land_cover = ee.ImageCollection("ESA/WorldCover/v100").first().clip(aoi).select(['Map'], ['landcover'])

# Infrastructure (Calculating distance to built-up areas/roads)
# Note: Using class 50 (Built-up) from WorldCover to estimate urban proximity
dist_roads = land_cover.eq(50).fastDistanceTransform().sqrt().multiply(1000).rename('dist_roads')

# Merging all bands into a single multi-dimensional image stack
predictors = bioclim.addBands(elevation).addBands(slope).addBands(land_cover).addBands(dist_roads).select([
    'bio01', 'bio05', 'bio06', 'bio07', 'bio12', 'bio13', 'bio14', 'bio15',
    'elevation', 'slope', 'landcover', 'dist_roads'
])

# SAMPLING
# Extracting environmental data for each presence point (Climate + Topography + Land Cover + Proximity)
sampled_presence_data = predictors.sampleRegions(
    collection=geospatial_points,
    properties=['year'],
    scale=1000,
    geometries=True
)

### MAP visualization of some useful environmental data (uncomment to see results)

In [ ]:

"""
predictors = bioclim.addBands(elevation).addBands(slope).addBands(land_cover).addBands(dist_roads).select([
    'bio01', 'bio05', 'bio06', 'bio07', 'bio12', 'bio13', 'bio14', 'bio15', 'elevation', 'slope', 'landcover', 'dist_roads'
])

# Temperature Palette (Bio01, Bio06, etc.) - From cold blue to heat red
vis_temp = {'min': -50, 'max': 300, 'palette': ['blue', 'cyan', 'green', 'yellow', 'red']}

# Precipitation Palette (Bio12, Bio14) - From dry white to rainfall blue
vis_precip = {'min': 0, 'max': 2500, 'palette': ['white', 'lightblue', 'blue', 'darkblue']}

# Elevation Palette
vis_el = {'min': 0, 'max': 2500, 'palette': ['006600', '002200', 'fff700', 'ab7634', 'c7d6ec', 'ffffff']}

# Slope Palette - From gentle green to steep red
vis_slope = {'min': 0, 'max': 45, 'palette': ['white', 'orange', 'red']}

# Official ESA WorldCover Palette (Landcover)
vis_lc = {'bands': ['landcover'], 'min': 10, 'max': 100,
          'palette': ['006400', 'ffbb22', 'ffff4c', 'f096ff', 'fa0000', 'b4b4b4', 'f0f0f0', '0064ff', '00bb88', 'ffff4c']}

# Distance to Roads Palette - From red (close) to white (far)
vis_dist = {'min': 0, 'max': 10000, 'palette': ['red', 'orange', 'white']}

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Adding layers one by one (toggle them in the layer menu)
Map.addLayer(predictors.select('bio01'), vis_temp, 'Annual Mean Temperature (Bio01)', False)
Map.addLayer(predictors.select('bio06'), vis_temp, 'Min Temperature of Coldest Month (Bio06)', False)
Map.addLayer(predictors.select('bio12'), vis_precip, 'Annual Precipitation (Bio12)', False)
Map.addLayer(predictors.select('elevation'), vis_el, 'Elevation (SRTM)', False)
Map.addLayer(predictors.select('slope'), vis_slope, 'Slope')
Map.addLayer(predictors.select('landcover'), vis_lc, 'Land Cover (ESA WorldCover)')
Map.addLayer(predictors.select('dist_roads'), vis_dist, 'Proximity to Urban/Road Areas')

# Adding presence points as a visual reference
Map.addLayer(geospatial_points, {'color': 'magenta'}, 'Actual Nests (Points)')

# Center the map
Map.centerObject(geospatial_points, 7)
Map


"""

### VIF

In this section, we call the **VIF_variable_selector** script to perform a feature selection. This process identifies the variables that best represent our data from the set of all potentially useful predictors we previously considered. Once we know the golden set there is no need to execute this, but you can uncomment it to see the results!

bio01, bio07, bio12, bio15, elevation, slope, dist_roads and landcover turn out to be the most representative variables as seen in the documentation.

In [ ]:
"""
from VIF_variable_selector import run_vif_cleaner

print("Downloading data from GEE and executing VIF...")
df_total = geemap.ee_to_df(sampled_presence_data)

# Setting the threshold to 10.0, though 5.0 could also be considered.
threshold = 10.0

print("Starting redundancy purge...")
final_variables, vif_report = run_vif_cleaner(df_total, threshold)

print("\n--- GOLDEN SUBSET ---")
print(f"Surviving variables: {final_variables}")
print("\nFinal VIF values:")
print(vif_report)
"""

This has been used to generate the graphs present in the documentation.
You can uncomment it too

In [ ]:
"""
import matplotlib.pyplot as plt

data = {
    'Variable': ['bio05', 'bio06', 'bio13', 'bio14', 'elevation', 'bio01', 'bio15', 'slope', 'bio07', 'bio12', 'dist_roads', 'landcover'],
    'VIF': [500, 104.91, 24.71, 10.16, 2.86, 2.36, 2.34, 1.80, 1.61, 1.35, 1.31, 1.11],
    'Subset': ['Dropped', 'Dropped', 'Dropped', 'Dropped', 'Golden', 'Golden', 'Golden', 'Golden', 'Golden', 'Golden', 'Golden', 'Golden']
}

df = pd.DataFrame(data)
# Invert the order so the Golden subset appears at the top of the horizontal chart
df = df.iloc[::-1]

# Red for Dropped, Blue for Golden
colors = ['#e74c3c' if s == 'Dropped' else '#3498db' for s in df['Subset']]

plt.figure(figsize=(10, 8))

# Create horizontal bar chart
bars = plt.barh(df['Variable'], df['VIF'], color=colors, edgecolor='black', alpha=0.8)

# Apply log scale to the X-axis for better visibility of low VIF values
plt.xscale('log')

# Add VIF value labels to the end of each bar
for bar in bars:
    xval = bar.get_width()
    plt.text(xval * 1.1, bar.get_y() + bar.get_height()/2,
             f'{xval if xval < 400 else "inf"}',
             va='center', ha='left', fontsize=10, fontweight='bold')

# Add vertical line at the VIF=10 threshold
plt.axvline(x=10, color='gray', linestyle='--', alpha=0.6, label='Threshold (VIF=10)')

plt.title('VIF Analysis: Feature Selection for Vespa velutina Model', fontsize=14, pad=20)
plt.xlabel('VIF Value (Log Scale)')
plt.ylabel('Environmental Predictors')

plt.grid(axis='x', linestyle=':', alpha=0.5)

# Custom legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color='#3498db', lw=4, label='Golden Subset (Accepted)'),
                   Line2D([0], [0], color='#e74c3c', lw=4, label='Redundant (Dropped)')]
plt.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

"""

### Feature Filtering: Final Presence Selection

In this section, we filter the presence records to retain only the specific environmental variables of interest for our model based on their VIF

In [ ]:
# Variables that maintained a VIF < 10 (FINAL PRESENCE COLLECTION)
golden_variables = [
    'elevation', 'bio01', 'bio15', 'slope',
    'bio07', 'bio12', 'dist_roads', 'landcover'
] # dist_roads sometimes causes bugs; we can re-insert it later if needed

# Filter the multi-band image to keep only the selected predictors
final_predictors = predictors.select(golden_variables)

# Sampling the final environmental data at presence locations
final_presence_data = final_predictors.sampleRegions(
    collection=geospatial_points,
    properties=['year'],
    scale=1000,
    geometries=True
)

# Labeling these records as class 1 (Presence)
final_presence_data = final_presence_data.map(lambda f: f.set('class', 1))

----------------------------------------------------------------

# Generating pseudoabsences and integration

## Pseudo-absence Generation Strategy 1: SM1 Random

We generate random points within our Area of Interest (AOI) that do not overlap with the known presence locations. Once generated, we visualize and save them into a dataset. We merge the presence and absence datasets into the total_dataset_SM1 dataset and we label both presences (1) and absences (0) so that our model can learn from it.

Predictor layers are re-instantiated and re-clipped to bypass persistent mask propagation inherited from earlier local-scale operations. This ensures that continental-scale sampling (SM1/SM2) draws from a complete, untruncated environmental pool, preventing niche truncation and ensuring model validity.

In [ ]:

# Following Barbet-Massin et al. (2012), we define a broad continental pool.
# We use a 2,000 km buffer and.bounds() to create a standard rectangular sampling frame.
sm1_aoi = geospatial_points.geometry().bounds().buffer(2000000)

# We must reload datasets to ensure the.clip() covers the entire 2,000 km area.
bioclim_sm1 = ee.Image('WORLDCLIM/V1/BIO').clip(sm1_aoi)
srtm_sm1 = ee.Image('USGS/SRTMGL1_003').clip(sm1_aoi)
slope_sm1 = ee.Terrain.slope(srtm_sm1).rename('slope')
land_cover_sm1 = ee.ImageCollection("ESA/WorldCover/v100").first().clip(sm1_aoi).select(['Map'], ['landcover'])

# Re-calculate Distance to Roads/Urban for the new extent
# Class 50 in ESA WorldCover represents 'Built-up' / Urban areas
dist_roads_sm1 = land_cover_sm1.eq(50).fastDistanceTransform().sqrt().multiply(1000).rename('dist_roads')

# Build the final predictor stack (we only include the golden variables)
sm1_predictors = bioclim_sm1.addBands(srtm_sm1.select('elevation')) \
                           .addBands(slope_sm1) \
                           .addBands(land_cover_sm1) \
                           .addBands(dist_roads_sm1) \
                           .select(golden_variables)


# Land mask: Class 80 in ESA WorldCover is permanent water
land_mask_sm1 = land_cover_sm1.neq(80)

# Exclusion mask 1km safety buffer around known nests to avoid false negatives
presence_mask_img = ee.Image().paint(geospatial_points, 1)
exclusion_mask_sm1 = presence_mask_img.fastDistanceTransform().sqrt().gt(1)
sampling_area_sm1 = ee.Image(1).clip(sm1_aoi).updateMask(exclusion_mask_sm1).updateMask(land_mask_sm1)


# We request 150,000 points (Oversampling) to ensure we reach the target after masking.
# We then.limit() to exactly match the number of presences for a 1:1 ratio.
target_count = 28887

pseudo_absences_raw = sm1_predictors.updateMask(sampling_area_sm1).sample(
    region=sm1_aoi,
    scale=1000,
    numPixels=150000,
    seed=67,
    geometries=True
)

# Truncate and Label as Class 0
absences_sm1 = pseudo_absences_raw.limit(target_count).map(lambda f: f.set('class', 0))

# Label presence data as Class 1 and merge
final_presence_data = final_presence_data.map(lambda f: f.set('class', 1))
total_dataset_SM1 = final_presence_data.merge(absences_sm1)

print(f"SM1 Dataset ready for training.")
print(f"Final Count: {absences_sm1.size().getInfo()} Absences (Class 0) | {final_presence_data.size().getInfo()} Presences (Class 1)")

The code below helps visualizing where the pseudoabsences have been located.

In [ ]:

# SM1 VISUALIZATION SECTION
Map = geemap.Map()
Map.add_basemap('HYBRID')

# Styling for presence and absence points
vespa_style = {'color': 'red', 'pointSize': 4, 'pointShape': 'circle', 'width': 1}
absences_style = {'color': '00ffff', 'pointSize': 4, 'pointShape': 'square', 'width': 1} # Cyan squares

# Visualization parameters for the sampling mask to see where points can be placed
vis_mask = {
    'min': 0,
    'max': 1,
    'palette': ['000000', 'yellow'],
    'opacity': 0.35
}

# Add layers to the map
Map.addLayer(sampling_area_sm1, vis_mask, 'Debug: Sampling Area (Allowed Zones)')
Map.addLayer(absences_sm1.style(**absences_style), {}, 'Pseudo-absences (Class 0)')
Map.addLayer(geospatial_points.style(**vespa_style), {}, 'Vespa Presences (Class 1)')

Map.centerObject(geospatial_points, 6)
Map


## Pseudo-absence Generation Strategy 2: SM2 Spatially Constrained
We generate random points within our area of interest that do not overlap with the presence locations, but ONLY within a close enough proximity to our existing points. This forces the model to learn from environments relatively near our study area. While its impact is somewhat limited due to the significant overlap with the overall AOI, and as we did in the SM1, we merge presences and absences.

In [ ]:
## SM2 ABSENCE GENERATION (SPATIALLY CONSTRAINED)


sm2_aoi = geospatial_points.geometry().buffer(350000)


bioclim_sm2 = ee.Image('WORLDCLIM/V1/BIO').clip(sm2_aoi)
srtm_sm2 = ee.Image('USGS/SRTMGL1_003').clip(sm2_aoi)
slope_sm2 = ee.Terrain.slope(srtm_sm2).rename('slope')
land_cover_sm2 = ee.ImageCollection("ESA/WorldCover/v100").first().clip(sm2_aoi).select(['Map'], ['landcover'])
dist_roads_sm2 = land_cover_sm2.eq(50).fastDistanceTransform().sqrt().multiply(1000).rename('dist_roads')

sm2_predictors = bioclim_sm2.addBands(srtm_sm2.select('elevation')) \
                          .addBands(slope_sm1) \
                          .addBands(land_cover_sm1) \
                          .addBands(dist_roads_sm1) \
                          .select(golden_variables)

presence_img = ee.Image().paint(geospatial_points, 1)
land_mask = land_cover_sm2.neq(80)
exclusion_mask = presence_img.fastDistanceTransform().sqrt().gt(1)
sampling_area_sm2 = ee.Image(1).clip(sm2_aoi).updateMask(exclusion_mask).updateMask(land_mask)

# We sample from 'sm2_predictors'
pseudo_absences_sm2_raw = sm2_predictors.updateMask(sampling_area_sm2).sample(
    region=sm2_aoi,
    scale=1000,
    numPixels=100000,
    seed=67,
    geometries=True
)


absences_sm2 = pseudo_absences_sm2_raw.limit(29306).map(lambda f: f.set('class', 0))
presences_sm2 = final_presence_data.map(lambda f: f.set('class', 1))
total_dataset_sm2 = presences_sm2.merge(absences_sm2)

print(f"SM2 Dataset fixed. Total instances: {total_dataset_sm2.size().getInfo()}")

In [ ]:

# SM2 VISUALIZATION SECTION

Map = geemap.Map()
Map.add_basemap('HYBRID')

vespa_style = {'color': 'red', 'pointSize': 4, 'pointShape': 'circle', 'width': 1}
absence_style = {'color': '00ffff', 'pointSize': 4, 'pointShape': 'square', 'width': 1}

# Search Mask Configuration (350km "Cloud")
# unmask(0) ensures the 1km exclusion gaps appear black
# clip(aoi_sm2) ensures we see the actual cloud shape rather than a bounding box
vis_mask = {
    'min': 0,
    'max': 1,
    'palette': ['000000', 'yellow'], # Black for restricted, Yellow for allowed
    'opacity': 0.35
}

Map.addLayer(sampling_area_sm2.unmask(0).clip(sm2_aoi), vis_mask, 'SM2 Sampling Region (350km Cloud)')
Map.addLayer(absences_sm2.style(**absence_style), {}, 'SM2 Pseudo-absences (Class 0)')
Map.addLayer(presences_sm2.style(**vespa_style), {}, 'Vespa Presences (Class 1)')

# Center the map on the invasion front
Map.centerObject(geospatial_points, 6)
Map

## SM3: OCVM to detect different climates

# Random Forest classifier


## SM1

In [ ]:
import geemap

classifier_sm1 = ee.Classifier.smileRandomForest(100).train(
    features = total_dataset_SM1,
    classProperty = 'class',
    inputProperties = golden_variables
)

# From white/yellow (low probability) to dark red (high probability)
vis_params = {
    'min': 0,
    'max': 1,
    'palette': ['#ffffff', '#fdfd96', '#ffb347', '#ff6961', '#8b0000']
}

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Adding the Habitat Suitability Map
# To see see continuous probabilities (0 to 1) instead of binary classification,
# we use the 'probability' mode
prob_classifier = classifier_sm1.setOutputMode('PROBABILITY')

# Classify the multi-band image using the final predictors
suitability_map = final_predictors.select(golden_variables).classify(prob_classifier)

"""
# Displaying the results
Map.addLayer(suitability_map, vis_params, 'Habitat Suitability Index (Probability)')
Map.addLayer(final_presence_data, {'color': 'blue'}, 'Observed Presences')

Map.centerObject(geospatial_points, 5)
Map
"""

In [ ]:
bioclim_global = ee.Image('WORLDCLIM/V1/BIO')
srtm_global = ee.Image('USGS/SRTMGL1_003')
landcover_global = ee.ImageCollection("ESA/WorldCover/v100").first()

elev_global = srtm_global.select('elevation')
slope_global = ee.Terrain.slope(srtm_global).rename('slope')
# Calculate distance to urban areas (class 50) across the entire map
dist_roads_global = landcover_global.eq(50).fastDistanceTransform().sqrt().multiply(1000).rename('dist_roads')

terrain_global = landcover_global.select(['Map'], ['landcover'])
predictors_global = bioclim_global.addBands(elev_global) \
                                   .addBands(slope_global) \
                                   .addBands(terrain_global) \
                                   .addBands(dist_roads_global)


# Apply .classify() to the global image.
# The 'classifier' already contains the trained knowledge.
map_fitness_final = predictors_global.select(golden_variables).classify(classifier_sm1.setOutputMode('PROBABILITY'))

Map = geemap.Map()
Map.add_basemap('HYBRID')
Map.addLayer(map_fitness_final, {'min': 0, 'max': 1, 'palette': ['white', 'yellow', 'red']}, 'Fitness Vespa Global')
Map

## SM2 PRUEBAS

In [ ]:
import geemap

# Train the Random Forest classifier for the SM2 model
# Using 100 trees and the spatially constrained dataset (total_dataset_sm2)
classifier_sm2 = ee.Classifier.smileRandomForest(100).train(
    features = total_dataset_sm2,
    classProperty = 'class',
    inputProperties = golden_variables
)

# Define a color palette: from white/yellow (low probability) to dark red (high probability)
vis_params = {
    'min': 0,
    'max': 1,
    'palette': ['#ffffff', '#fdfd96', '#ffb347', '#ff6961', '#8b0000']
}

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Switching the SM2 classifier to PROBABILITY mode
# This creates a continuous Habitat Suitability Index (HSI)
prob_classifier_sm2 = classifier_sm2.setOutputMode('PROBABILITY')

# Classify the multi-band predictor image using the SM2 model
suitability_map_sm2 = final_predictors.select(golden_variables).classify(prob_classifier_sm2)

"""
# Visualizing the SM2 results
Map.addLayer(suitability_map_sm2, vis_params, 'Habitat Suitability Index (SM2 - Constrained)')

# Overlaying observed presence points for visual validation
Map.addLayer(final_presence_data, {'color': 'blue'}, 'Observed Presences')

# Center the map on the study area
Map.centerObject(geospatial_points, 5)
Map
"""


In [ ]:
# Preparar predictores globales con nombres explícitos
bioclim_global = ee.Image('WORLDCLIM/V1/BIO')
srtm_global = ee.Image('USGS/SRTMGL1_003').select(['elevation'])
land_cover_img = ee.ImageCollection("ESA/WorldCover/v100").first()

slope_global = ee.Terrain.slope(srtm_global).rename('slope')
landcover_global = land_cover_img.select(['Map'], ['landcover'])
dist_roads_global = landcover_global.eq(50).fastDistanceTransform().sqrt().multiply(1000).rename('dist_roads')

global_predictors = bioclim_global \
    .addBands(srtm_global) \
    .addBands(slope_global) \
    .addBands(landcover_global) \
    .addBands(dist_roads_global)

map_fitness_final_sm2 = global_predictors.select(golden_variables) \
    .classify(classifier_sm2.setOutputMode('PROBABILITY'))

# 5. Visualización
Map = geemap.Map()
Map.add_basemap('HYBRID')
Map.addLayer(map_fitness_final_sm2, {'min': 0, 'max': 1, 'palette': ['white', 'yellow', 'red']}, 'Fitness Vespa SM2 (Global)')
Map.centerObject(geospatial_points, 5)
Map

# How good are our models?

# REVISIONES

Cosas a arreglar o revisar:

5- ENTRENAR EL MODELO, LUEGO HACER VALIDACIÓN CRUZADA (SE VERÁ COMO SE HACE) PARA VER CÓMO DE BUENO ES.

6- REVISAR, DOCUMENTAR, README DE GITHUB


Enlaces útiles:

https://ramirodcrego.com/teaching/gee/

https://developers.google.com/earth-engine/tutorials/community/species-distribution-modeling?hl=es-419

https://www.tandfonline.com/doi/full/10.1080/10095020.2025.2546507#abstract

https://www.researchgate.net/publication/284246225_A_multi-scale_approach_to_identify_invasion_drivers_and_invaders'_future_dynamics

https://onlinelibrary.wiley.com/doi/full/10.1002/ece3.70029

https://besjournals.onlinelibrary.wiley.com/doi/10.1111/j.2041-210X.2011.00172.x

https://biogeography-usc.org/positive/Prediction.html

https://www.sciencedirect.com/science/article/abs/pii/S0006320711001315

https://revistaecosistemas.net/index.php/ecosistemas/article/view/2987/1962

https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0071218 (PAPER SM4)

https://gidahatari.com/ih-es/bioclim-un-sistema-de-analisis-y-prediccion-de-bioclimas (BIOCLIM)

https://www.researchgate.net/publication/226562656_DOMAIN_a_flexible_modelling_procedure_for_mapping_potential_distributions_of_plants_and_animals (DOMAIN)

https://www.sciencedirect.com/science/article/pii/S0304380011000780 (PAPER 1 JUSTIFICACIÓN BUFFER)

https://besjournals.onlinelibrary.wiley.com/doi/10.1111/j.2041-210X.2010.00036.x (PAPER 2 JUSTIFICACIÓN BUFFER)

https://besjournals.onlinelibrary.wiley.com/doi/full/10.1111/1365-2664.12724 (PAPER 3 JUSTIFICACIÓN BUFFER)

https://www.plant-animal.es/pdfs/Herrera.2024.Ecosistemas.pdf (paper justificación presencias)